# 00 — Vimeo-90k Data Preparation
Download Vimeo triplet test set, generate events with v2e (SuperSloMo enabled), split into train/val/test, and save to Drive.

**Processing strategy**: ~3,252 triplets processed in configurable batches (~300/session). Resumable via skip-if-exists.

In [ ]:
# ── Cell 1: Setup & Mount ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/493Project')

%pip install -q torchmetrics

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 45.2 MB/s eta 0:00:00


In [ ]:
# ── Cell 2: Download Vimeo Triplet Test Set ──────────────────────────────────
import os

VIMEO_ZIP = '/content/vimeo_interp_test.zip'
VIMEO_DIR = '/content/vimeo_interp_test'

if not os.path.exists(VIMEO_DIR):
    if not os.path.exists(VIMEO_ZIP):
        !wget -q http://data.csail.mit.edu/tofu/testset/vimeo_interp_test.zip -O {VIMEO_ZIP}
        print(f'Downloaded {os.path.getsize(VIMEO_ZIP) / 1e9:.2f} GB')
    !unzip -q {VIMEO_ZIP} -d /content/
    print('Extracted.')
else:
    print('Vimeo test set already extracted.')

# Verify structure
sample = os.path.join(VIMEO_DIR, 'target')
seqs = sorted(os.listdir(sample))
print(f'Sequence folders: {len(seqs)} (first 5: {seqs[:5]})')
total = sum(len(os.listdir(os.path.join(sample, s))) for s in seqs)
print(f'Total triplets: {total}')

Downloaded 2.99 GB
Extracted.
Sequence folders: 78 (first 5: ['00001', '00002', '00003', '00004', '00005'])
Total triplets: 3782


In [ ]:
# ── Cell 3: Clone & install v2e + download SuperSloMo checkpoint ─────────────
import os

if not os.path.exists('/content/v2e'):
    !git clone https://github.com/SensorsINI/v2e /content/v2e

%cd /content/v2e
%pip install -q .
%pip install -q gdown
%cd /content/

# Download SuperSloMo checkpoint from Google Drive (151 MB)
# This is v2e's custom grayscale-retrained SuperSloMo model
SLOMO_CKPT = '/content/v2e/input/SuperSloMo39.ckpt'
if not os.path.exists(SLOMO_CKPT) or os.path.getsize(SLOMO_CKPT) < 1_000_000:
    os.makedirs('/content/v2e/input', exist_ok=True)
    !gdown "1ETID_4xqLpRBrRo1aOT7Yphs3QqWR_fx" -O {SLOMO_CKPT}
    print(f'Downloaded SuperSloMo checkpoint: {os.path.getsize(SLOMO_CKPT) / 1e6:.1f} MB')
else:
    print(f'SuperSloMo checkpoint already exists ({os.path.getsize(SLOMO_CKPT) / 1e6:.1f} MB)')

assert os.path.isfile(SLOMO_CKPT) and os.path.getsize(SLOMO_CKPT) > 1_000_000, \
    f'Checkpoint missing or too small: {SLOMO_CKPT}'
print('v2e + SuperSloMo ready.')

Cloning into '/content/v2e'...
remote: Enumerating objects: 3334, done.
remote: Counting objects: 100% (968/968), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 3334 (delta 927), reused 878 (delta 878), pack-reused 2366 (from 2)
Receiving objects: 100% (3334/3334), 34.36 MiB | 43.97 MiB/s, done.
Resolving deltas: 100% (2448/2448), done.
/content/v2e
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 11.9 MB/s eta 0:00:00
/content
Downloading...
From (original): https://drive.google.com/uc?id=1ETID_4xqLpRBrRo1aOT7Yphs3QqWR_fx
From (redirected): https://drive.google.com/uc?id=1ETID_4xqLpRBrRo1aOT7Yphs3QqWR_fx

In [ ]:
# ── Cell 4: Build full triplet list, shuffle, split, and save lists to Drive ─
import os
import random

VIMEO_DIR   = '/content/vimeo_interp_test'
TARGET_DIR  = os.path.join(VIMEO_DIR, 'target')
DRIVE_ROOT  = '/content/drive/MyDrive/493Project/data/vimeo'
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Build list of all triplet relative paths (e.g. "00001/0266")
all_triplets = []
for seq in sorted(os.listdir(TARGET_DIR)):
    seq_dir = os.path.join(TARGET_DIR, seq)
    if not os.path.isdir(seq_dir):
        continue
    for clip in sorted(os.listdir(seq_dir)):
        clip_dir = os.path.join(seq_dir, clip)
        # Verify all 3 frames exist
        if (os.path.isfile(os.path.join(clip_dir, 'im1.png')) and
            os.path.isfile(os.path.join(clip_dir, 'im2.png')) and
            os.path.isfile(os.path.join(clip_dir, 'im3.png'))):
            all_triplets.append(f'{seq}/{clip}')

print(f'Total valid triplets: {len(all_triplets)}')

# Shuffle and split 80/10/10
rng = random.Random(42)
rng.shuffle(all_triplets)

n = len(all_triplets)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)
train_list = all_triplets[:n_train]
val_list   = all_triplets[n_train : n_train + n_val]
test_list  = all_triplets[n_train + n_val:]

print(f'Train: {len(train_list)} | Val: {len(val_list)} | Test: {len(test_list)}')

# Save split lists to Drive
for name, lst in [('tri_trainlist.txt', train_list),
                  ('tri_vallist.txt',   val_list),
                  ('tri_testlist.txt',  test_list)]:
    path = os.path.join(DRIVE_ROOT, name)
    with open(path, 'w') as f:
        f.write('\n'.join(lst) + '\n')
    print(f'Saved {path} ({len(lst)} entries)')

print('\nSplit lists saved to Drive. These are permanent — processing order follows this list.')

Total valid triplets: 3782
Train: 3025 | Val: 378 | Test: 379
Saved /content/drive/MyDrive/493Project/data/vimeo/tri_trainlist.txt (3025 entries)
Saved /content/drive/MyDrive/493Project/data/vimeo/tri_vallist.txt (378 entries)
Saved /content/drive/MyDrive/493Project/data/vimeo/tri_testlist.txt (379 entries)

Split lists saved to Drive. These are permanent — processing order follows this list.


In [ ]:
# ── Cell 5b: Process the batch (run after Cell 5 test passes) ────────────────
import os, subprocess, shutil, torch, time, sys
sys.path.insert(0, '/content/drive/MyDrive/493Project')
from src.data import load_h5_events, events_to_voxel_grid, normalize_voxel

VIMEO_DIR   = '/content/vimeo_interp_test'
TARGET_DIR  = os.path.join(VIMEO_DIR, 'target')
DRIVE_ROOT  = '/content/drive/MyDrive/493Project/data/vimeo'
TEMP_DIR    = '/content/temp_v2e'
SLOMO_CKPT  = '/content/v2e/input/SuperSloMo39.ckpt'
VOXEL_W, VOXEL_H, NUM_BINS = 448, 256, 5

# Re-use BATCH_SIZE and START_FROM from Cell 5
all_triplets = []
for sf in ['tri_trainlist.txt', 'tri_vallist.txt', 'tri_testlist.txt']:
    with open(os.path.join(DRIVE_ROOT, sf)) as f:
        all_triplets.extend([l.strip() for l in f if l.strip()])
batch = all_triplets[START_FROM : START_FROM + BATCH_SIZE]
print(f'Processing triplets {START_FROM} to {START_FROM + len(batch) - 1}')

os.makedirs(TEMP_DIR, exist_ok=True)
done, skipped, errors = 0, 0, 0
t_start = time.time()

for i, rel_path in enumerate(batch):
    src_dir   = os.path.join(TARGET_DIR, rel_path)
    dst_dir   = os.path.join(DRIVE_ROOT, 'processed', rel_path)
    voxel_dst = os.path.join(dst_dir, 'voxel.pt')

    if os.path.exists(voxel_dst):
        skipped += 1
        continue

    os.makedirs(dst_dir, exist_ok=True)
    temp_out = os.path.join(TEMP_DIR, rel_path.replace('/', '_'))

    try:
        for fname in ['im1.png', 'im2.png', 'im3.png']:
            src = os.path.join(src_dir, fname)
            dst = os.path.join(dst_dir, fname)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)

        os.makedirs(temp_out, exist_ok=True)
        t0 = time.time()
        # v2e exits code 1 due to cv2.destroyAllWindows() bug — ignore return code,
        # check events.h5 existence instead
        subprocess.run([
            'v2e', '-i', src_dir,
            '--output_folder', temp_out,
            '--input_frame_rate', '30',
            '--slomo_model', SLOMO_CKPT,
            '--pos_thres', '0.2', '--neg_thres', '0.2',
            '--sigma_thres', '0.03',
            '--refractory_period', '0.0005',
            '--skip_video_output',
            '--dvs_h5', 'events.h5',
            '--no_preview',
            '--dvs_params', 'noisy',
        ], capture_output=True, text=True, timeout=300)
        v2e_time = time.time() - t0

        h5_path = os.path.join(temp_out, 'events.h5')
        if not os.path.exists(h5_path) or os.path.getsize(h5_path) == 0:
            print(f'  [{START_FROM + i}] {rel_path} — no events.h5 ({v2e_time:.0f}s)')
            errors += 1
            continue

        events = load_h5_events(h5_path)
        if events is None or events.shape[0] == 0:
            voxel = torch.zeros(NUM_BINS, VOXEL_H, VOXEL_W, dtype=torch.float32)
        else:
            voxel = events_to_voxel_grid(events, num_bins=NUM_BINS,
                                          width=VOXEL_W, height=VOXEL_H)
            voxel = normalize_voxel(voxel)

        torch.save(voxel, voxel_dst)
        done += 1

        # Print every 10 triplets
        if done % 10 == 0:
            elapsed = time.time() - t_start
            rate = (done + skipped) / elapsed * 3600
            remaining = (len(batch) - done - skipped - errors) / max(rate / 3600, 1e-9)
            print(f'  [{START_FROM + i}] {done} done, {skipped} skip, {errors} err | '
                  f'{v2e_time:.0f}s/clip | ~{remaining/60:.0f}min remaining')

    except subprocess.TimeoutExpired:
        print(f'  [{START_FROM + i}] {rel_path} — TIMEOUT (>300s)')
        errors += 1
        if os.path.exists(voxel_dst):
            os.remove(voxel_dst)
    except Exception as e:
        print(f'  [{START_FROM + i}] {rel_path} — ERROR: {e}')
        errors += 1
        if os.path.exists(voxel_dst):
            os.remove(voxel_dst)
    finally:
        if os.path.exists(temp_out):
            shutil.rmtree(temp_out, ignore_errors=True)

if os.path.exists(TEMP_DIR):
    shutil.rmtree(TEMP_DIR, ignore_errors=True)

total_time = time.time() - t_start
print(f'\n{"="*60}')
print(f'Batch complete in {total_time/60:.1f} min')
print(f'  {done} processed, {skipped} skipped, {errors} errors')
print(f'Next session: set START_FROM = {START_FROM + len(batch)}')

Processing triplets 900 to 1899
  [909] 10 done, 0 skip, 0 err | 20s/clip | ~321min remaining
  [919] 20 done, 0 skip, 0 err | 23s/clip | ~345min remaining
  [929] 30 done, 0 skip, 0 err | 18s/clip | ~339min remaining
  [939] 40 done, 0 skip, 0 err | 23s/clip | ~353min remaining
  [949] 50 done, 0 skip, 0 err | 19s/clip | ~356min remaining
  [959] 60 done, 0 skip, 0 err | 23s/clip | ~355min remaining
  [969] 70 done, 0 skip, 0 err | 19s/clip | ~354min remaining
  [979] 80 done, 0 skip, 0 err | 20s/clip | ~352min remaining
  [989] 90 done, 0 skip, 0 err | 18s/clip | ~343min remaining
  [999] 100 done, 0 skip, 0 err | 19s/clip | ~338min remaining
  [1009] 110 done, 0 skip, 0 err | 39s/clip | ~336min remaining
  [1019] 120 done, 0 skip, 0 err | 26s/clip | ~334min remaining
  [1029] 130 done, 0 skip, 0 err | 19s/clip | ~328min remaining
  [1039] 140 done, 0 skip, 0 err | 17s/clip | ~323min remaining
  [1049] 150 done, 0 skip, 0 err | 18s/clip | ~320min remaining
  [1059] 160 done, 0 skip, 

In [ ]:
# ── Cell 6: Sanity check — visualize first 5 processed triplets ──────────────
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
import os

DRIVE_ROOT = '/content/drive/MyDrive/493Project/data/vimeo'

# Find first 5 triplets that have been processed
processed = []
for split_file in ['tri_trainlist.txt', 'tri_vallist.txt', 'tri_testlist.txt']:
    with open(os.path.join(DRIVE_ROOT, split_file)) as f:
        for line in f:
            rel = line.strip()
            d = os.path.join(DRIVE_ROOT, 'processed', rel)
            if os.path.isfile(os.path.join(d, 'voxel.pt')):
                processed.append(d)
            if len(processed) >= 5:
                break
        if len(processed) >= 5:
            break

print(f'Showing {len(processed)} processed triplets')

fig, axes = plt.subplots(len(processed), 4, figsize=(20, 4.5 * len(processed)))
if len(processed) == 1:
    axes = axes[np.newaxis, :]

col_titles = ['Frame f0 (im1)', 'Frame f1 (im3)', 'Ground Truth (im2)', 'Event Voxel (sum of bins)']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=13, fontweight='bold')

for row, d in enumerate(processed):
    f0  = Image.open(os.path.join(d, 'im1.png')).convert('RGB')
    f1  = Image.open(os.path.join(d, 'im3.png')).convert('RGB')
    gt  = Image.open(os.path.join(d, 'im2.png')).convert('RGB')
    voxel = torch.load(os.path.join(d, 'voxel.pt'), weights_only=True)

    axes[row, 0].imshow(f0); axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
    axes[row, 1].imshow(f1); axes[row, 1].set_xticks([]); axes[row, 1].set_yticks([])
    axes[row, 2].imshow(gt); axes[row, 2].set_xticks([]); axes[row, 2].set_yticks([])

    evt_sum = voxel.sum(dim=0).numpy()
    abs_max = max(abs(evt_sum.min()), abs(evt_sum.max()), 1e-8)
    axes[row, 3].imshow(evt_sum, cmap='RdBu_r', vmin=-abs_max, vmax=abs_max)
    axes[row, 3].set_xticks([]); axes[row, 3].set_yticks([])

    density = (voxel.abs() > 0.01).float().mean().item() * 100
    axes[row, 0].set_ylabel(
        f'{os.path.basename(os.path.dirname(d))}/{os.path.basename(d)}\n'
        f'voxel: {tuple(voxel.shape)}\n'
        f'range: [{voxel.min():.2f}, {voxel.max():.2f}]\n'
        f'active: {density:.1f}%',
        fontsize=8
    )

plt.suptitle('Vimeo-90k Event Voxel Sanity Check', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

# Per-bin stats for first sample
v = torch.load(os.path.join(processed[0], 'voxel.pt'), weights_only=True)
print(f'\nVoxel shape: {v.shape}')
for b in range(v.shape[0]):
    bv = v[b]
    print(f'  Bin {b}: min={bv.min():.3f}  max={bv.max():.3f}  '
          f'mean={bv.mean():.4f}  std={bv.std():.4f}  '
          f'nonzero={(bv.abs() > 0.01).sum().item()}/{bv.numel()}')

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ── Cell 7: Verification — count processed triplets per split ────────────────
import os

DRIVE_ROOT = '/content/drive/MyDrive/493Project/data/vimeo'

for split_name in ['tri_trainlist.txt', 'tri_vallist.txt', 'tri_testlist.txt']:
    path = os.path.join(DRIVE_ROOT, split_name)
    with open(path) as f:
        entries = [line.strip() for line in f if line.strip()]

    total = len(entries)
    processed = 0
    for rel in entries:
        d = os.path.join(DRIVE_ROOT, 'processed', rel)
        if os.path.isfile(os.path.join(d, 'voxel.pt')):
            processed += 1

    pct = 100 * processed / total if total > 0 else 0
    print(f'{split_name:25s}  {processed:>5}/{total:>5}  ({pct:.1f}%)')

# Spot-check a random sample
import random, torch
all_processed = []
for split_name in ['tri_trainlist.txt', 'tri_vallist.txt', 'tri_testlist.txt']:
    with open(os.path.join(DRIVE_ROOT, split_name)) as f:
        for line in f:
            rel = line.strip()
            d = os.path.join(DRIVE_ROOT, 'processed', rel)
            if os.path.isfile(os.path.join(d, 'voxel.pt')):
                all_processed.append(d)

if all_processed:
    sample_dir = random.choice(all_processed)
    voxel = torch.load(os.path.join(sample_dir, 'voxel.pt'), weights_only=True)
    from PIL import Image
    im1 = Image.open(os.path.join(sample_dir, 'im1.png'))
    print(f'\nRandom sample: {sample_dir}')
    print(f'  Frame size: {im1.size} (W×H)')
    print(f'  Voxel shape: {tuple(voxel.shape)} (C×H×W)')
    assert voxel.shape == (5, 256, 448), f'Unexpected voxel shape: {voxel.shape}'
    print('  Shape OK!')
else:
    print('No processed triplets found yet.')

tri_trainlist.txt            300/ 3025  (9.9%)
tri_vallist.txt                0/  378  (0.0%)
tri_testlist.txt               0/  379  (0.0%)

Random sample: /content/drive/MyDrive/493Project/data/vimeo/processed/00019/0178
  Frame size: (448, 256) (W×H)
  Voxel shape: (5, 256, 448) (C×H×W)
  Shape OK!
